In [1]:
import sys
import os

# Get the absolute path to the project directory
project_dir = os.path.abspath("..")

# Append the project directory to sys.path
if project_dir not in sys.path:
    sys.path.append(project_dir)
    
from src.predictionModule.LoadupSamples import LoadupSamples
from src.predictionModule.FilterSamples import FilterSamples
from src.predictionModule.MachineModels import MachineModels

import numpy as np
import pandas as pd
import datetime
import optuna
import random
import torch
import copy

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import logging
formatted_date = datetime.datetime.now().strftime("%d%b%y_%H%M").lower()

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter(fmt="%(asctime)s - %(message)s")
handler.setFormatter(formatter)
if not logger.hasHandlers():
    logger.addHandler(handler)
else:
    logger.handlers[:] = [handler]

#Output File handler
formatted_str = f"notebook-stomp-{formatted_date}"
file_handler = logging.FileHandler(f"{formatted_str}.log", mode="w")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Usage
logger.setLevel(logging.INFO)
logger.info("This will print to the notebook's output cell")

2025-09-10 17:09:46,481 - This will print to the notebook's output cell


c:\Users\KILightTouch\Desktop\RandomOdyssey\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
params = {
    "idxAfterPrediction": 5,
    'timesteps': 90,
    'target_option': 'last',
    "LoadupSamples_time_scaling_stretch": True,
    "LoadupSamples_time_inc_factor": 1,

    "FilterSamples_q_up": 0.6,
    
    "FilterSamples_cat_over20": True,
    "FilterSamples_cat_posOneYearReturn": False,
    "FilterSamples_cat_posFiveYearReturn": False,

    "LSTM_val_split": 0.1,
}

In [3]:
timegroup = "group_regOHLCV_over5years"
treegroup = "group_debug"

eval_date = datetime.date(year=2025, month=7, day=13)
evaldates = [eval_date - datetime.timedelta(days=i) for i in range(1, 6)]
start_train_date = datetime.date(year=2019, month=1, day=1)
split_Date = datetime.date(year=2025, month=1, day=1)
ls = LoadupSamples(
    train_start_date=start_train_date,
    test_dates=evaldates,
    treegroup=treegroup,
    timegroup=timegroup,
    params=params,
)
ls.load_samples(main_path = "../src/featureAlchemy/bin/")
ls.split_dataset(
    start_date=start_train_date,
    last_train_date=split_Date,
    last_test_date=eval_date
)
fs_pre = FilterSamples(
    Xtree_train = ls.train_Xtree, 
    ytree_train = ls.train_ytree, 
    treenames   = ls.featureTreeNames,
    Xtree_test  = ls.test_Xtree,  
    ytree_test  = ls.test_ytree,
    meta_train  = ls.meta_pl_train, 
    meta_test   = ls.meta_pl_test, 
    params      = params
)
mask_train_pre, mask_test_pre = fs_pre.categorical_masks()
ls.apply_masks(mask_train_pre, mask_test_pre)

2025-09-10 17:09:56,886 - Test date 2025-07-12 not found in the database. Omitting.


In [4]:
Xtree_train = ls.train_Xtree
ytree_train = ls.train_ytree
Xtree_test  = ls.test_Xtree
ytree_test  = ls.test_ytree

Xtime_train = ls.train_Xtime
ytime_train = ls.train_ytime
Xtime_test  = ls.test_Xtime
ytime_test  = ls.test_ytime

treenames   = ls.featureTreeNames
timenames   = ls.featureTimeNames
meta_train  = ls.meta_pl_train
meta_test   = ls.meta_pl_test

dates_tr = meta_train['date'].unique().sort()
dates_te = meta_test['date'].unique().sort()

from src.common.DataFrameTimeOperations import DataFrameTimeOperations as dfta
dates_tr_idx = dfta(meta_train, 'date').getNextLowerOrEqualIndices(dates_tr)
dates_te_idx = dfta(meta_test, 'date').getNextLowerOrEqualIndices(dates_te)

assert not any([i == -1 for i in dates_tr_idx])
assert not any([i == -1 for i in dates_te_idx])

In [ ]:
# ---- knobs (use existing globals if present) ----
device = "cuda" if torch.cuda.is_available() else "cpu"
n_splits = 100
n_test_days = 60

assert "Xtime_train" in globals() and "ytree_train" in globals(), "Need Xtime_train/ytree_train"
nS, nT, nF = Xtime_train.shape

In [6]:
def geometric_mean_safe(arr):
    arr = np.asarray(arr, dtype=float)
    minv = np.min(arr) if arr.size else 0.0
    shift = -minv + 1e-9 if minv <= 0 else 0.0
    return float(np.exp(np.mean(np.log(arr + shift)))) if arr.size else np.nan

def metric(arr):
    """Custom cluster score function."""
    gm = geometric_mean_safe(arr)
    return gm - 1

def make_design(X, t_win, feat_idx):
    feat_idx = np.atleast_1d(feat_idx)
    Xw = X[:, -t_win:, feat_idx]
    return Xw.reshape(Xw.shape[0], -1)

def _score_once_lstm(
    t_win: int,
    k: int,
    f_idcs: list[int],
    Xtr: np.ndarray,
    ytr_tree: np.ndarray,
    ytr_time: np.ndarray,
    Xte: np.ndarray,
    yte_tree: np.ndarray,
    yte_time: np.ndarray,
    mm: MachineModels,
    quantile_val: float = 0.9,
    do_transform: bool = False,
    device: str = "cpu",
    min_cluster_train: int = 50,
) -> float:
    """
    Cluster train window, train one LSTM per cluster, pick cluster with lowest RMSE,
    predict on the paired test samples in that cluster, and select the highest
    predicted entries via a quantile threshold.
    """

    if k >= Xtr.shape[0]:
        return -np.inf

    # design matrices
    Xd_tr = make_design(Xtr, t_win, f_idcs)
    Xd_te = make_design(Xte, t_win, f_idcs)

    if do_transform:
        scaler = StandardScaler().fit(Xd_tr)
        Xd_tr = scaler.transform(Xd_tr)
        Xd_te = scaler.transform(Xd_te)

    n_feat = len(np.atleast_1d(f_idcs))
    Xseq_tr = Xd_tr.reshape(-1, t_win, n_feat)
    Xseq_te = Xd_te.reshape(-1, t_win, n_feat)

    km = KMeans(n_clusters=k, n_init="auto", random_state=0, init="k-means++").fit(Xd_tr)
    lab_tr = km.labels_
    lab_te = km.predict(Xd_te)

    best_c, best_model, best_rmse = None, None, float("inf")
    for c in range(k):
        mask_tr = (lab_tr == c)
        mask_te = (lab_te == c)
        if mask_tr.sum() < min_cluster_train:
            logger.info(f"[LSTM] cluster {c}: skipped (train size {mask_tr.sum()} < {min_cluster_train})")
            continue

        Xc_tr = Xseq_tr[mask_tr]
        yc_tr = ytr_time[mask_tr]

        try:
            logger.disabled = True
            model_c, info = mm.run_LSTM_torch(
                X_train=Xc_tr,
                y_train=yc_tr,
                X_test=None,
                y_test=None,
                device=device,
                logger_disabled=True,
            )
            logger.disabled = False
            val_rmse = float(info.get("val_rmse", float("inf")))
            logger.info(f"[LSTM] cluster {c}: train={mask_tr.sum()}, val_rmse={val_rmse:.6f}")

            if np.isfinite(val_rmse) and val_rmse < best_rmse:
                best_c, best_model, best_rmse = c, model_c, val_rmse
        except Exception as e:
            logger.disabled = False
            logger.warning(f"[LSTM] cluster {c}: training failed — {e}")
            continue
        finally:
            logger.disabled = False

    if best_c is None or best_model is None:
        logger.warning("[LSTM] no trainable cluster found.")
        return metric(1.0)

    # predict on test members of best cluster
    mask_te_local = (lab_te == best_c)
    if mask_te_local.sum() == 0:
        logger.warning(f"[LSTM] no best test cluster because cluster {best_c} has no test members.")
        return metric(1.0)

    preds = mm.predict_LSTM_torch(best_model, Xseq_te[mask_te_local], device=device)
    if preds.size == 0 or not np.all(np.isfinite(preds)):
        logger.warning(f"[LSTM] no prediction generated.")
        return metric(1.0)

    thr = float(np.quantile(preds, quantile_val))
    mask = preds >= thr
    y_selected = yte_tree[mask_te_local][mask]
    y_selected = y_selected[np.isfinite(y_selected)] 

    if y_selected.size == 0:
        logger.warning(f"[LSTM] no selected testing values.")
        return metric(1.0)

    score = metric(y_selected)

    logger.info(
        f"[LSTM] t_win={t_win}, k={k}, tr_size={Xd_tr.shape[0]}, te_size={Xd_te.shape[0]} "
        f"-> best_c={best_c}, best_val_rmse={best_rmse:.6f}, "
        f"cluster_te={mask_te_local.sum()}, thr@q={quantile_val}={thr}, score={score}"
    )

    return float(score) if np.isfinite(score) else metric(1.0)

In [7]:
max_training_days = 1200
N = len(dates_tr_idx)
lo = max_training_days - 1                      # min pivot (last train index)
hi = N - n_test_days - 1                      # max pivot
eligible = list(range(lo, hi + 1))
assert len(eligible) >= n_splits, f"Too few eligible pivots ({len(eligible)}) for n_splits={n_splits}"
pivots = sorted(random.sample(eligible, n_splits))

tr_slice_list = [slice(p - max_training_days + 1, p + 1) for p in pivots]      # [..)
te_slice_list = [slice(p + 1, p + 1 + n_test_days) for p in pivots]          # [..)

for p in pivots:
    logger.info(f"  Pivot {p}: Date {dates_tr[p]}")

2025-09-10 17:09:59,440 -   Pivot 1199: Date 2023-10-06
2025-09-10 17:09:59,440 -   Pivot 1200: Date 2023-10-09
2025-09-10 17:09:59,440 -   Pivot 1201: Date 2023-10-10
2025-09-10 17:09:59,440 -   Pivot 1215: Date 2023-10-30
2025-09-10 17:09:59,444 -   Pivot 1216: Date 2023-10-31
2025-09-10 17:09:59,444 -   Pivot 1220: Date 2023-11-06
2025-09-10 17:09:59,444 -   Pivot 1230: Date 2023-11-20
2025-09-10 17:09:59,446 -   Pivot 1231: Date 2023-11-21
2025-09-10 17:09:59,446 -   Pivot 1241: Date 2023-12-06
2025-09-10 17:09:59,448 -   Pivot 1243: Date 2023-12-08
2025-09-10 17:09:59,448 -   Pivot 1246: Date 2023-12-13
2025-09-10 17:09:59,448 -   Pivot 1249: Date 2023-12-18
2025-09-10 17:09:59,448 -   Pivot 1259: Date 2024-01-03
2025-09-10 17:09:59,448 -   Pivot 1261: Date 2024-01-05
2025-09-10 17:09:59,448 -   Pivot 1263: Date 2024-01-09
2025-09-10 17:09:59,448 -   Pivot 1264: Date 2024-01-10
2025-09-10 17:09:59,448 -   Pivot 1270: Date 2024-01-19
2025-09-10 17:09:59,453 -   Pivot 1271: Date 202

2025-09-10 17:09:59,468 -   Pivot 1457: Date 2024-10-16
2025-09-10 17:09:59,468 -   Pivot 1462: Date 2024-10-23
2025-09-10 17:09:59,470 -   Pivot 1464: Date 2024-10-25
2025-09-10 17:09:59,471 -   Pivot 1465: Date 2024-10-28
2025-09-10 17:09:59,471 -   Pivot 1467: Date 2024-10-30
2025-09-10 17:09:59,473 -   Pivot 1477: Date 2024-11-13
2025-09-10 17:09:59,474 -   Pivot 1484: Date 2024-11-22
2025-09-10 17:09:59,475 -   Pivot 1485: Date 2024-11-25
2025-09-10 17:09:59,476 -   Pivot 1486: Date 2024-11-26


In [ ]:
def make_objective():
    def objective(trial: optuna.Trial) -> float:
        # search space
        t_win = trial.suggest_int("t_win", 10, 60, step=5)
        k = trial.suggest_int("CLUSTERS", 4, 10, step=2)
        n_training_days = 1000
        do_transform = False
        f_idcs_cat = 2
        quantile_val = 0.8

        opt_params = copy.deepcopy(params)
        opt_params["idxAfterPrediction"] = 5
        opt_params["LoadupSamples_time_inc_factor"] = trial.suggest_int("LoadupSamples_time_inc_factor", 1, 81, step=10)
        opt_params["LSTM_units"] = trial.suggest_categorical("LSTM_units", [8, 16, 32])
        opt_params["LSTM_num_layers"] = trial.suggest_int("LSTM_num_layers", 1, 2)
        opt_params["LSTM_learning_rate"] = trial.suggest_float("LSTM_learning_rate", 1e-5, 1e-2, log=True)
        opt_params["LSTM_epochs"] = 6
        opt_params["LSTM_l1"] = 1e-5
        opt_params["LSTM_l2"] = 1e-5
        opt_params["LSTM_dropout"] = trial.suggest_float("LSTM_dropout", 1e-4, 1e-1, log=True)
        opt_params["LSTM_inter_dropout"] = trial.suggest_float("LSTM_inter_dropout", 1e-4, 1e-1, log=True)
        opt_params["LSTM_recurrent_dropout"] = trial.suggest_float("LSTM_recurrent_dropout", 1e-4, 1e-1, log=True)
        opt_params["LSTM_conv1d_kernel_size"] = 3
        opt_params["is_single_feature"] = False


        if f_idcs_cat == 0:       f_idcs = [0]
        elif f_idcs_cat == 1:     f_idcs = [1]
        elif f_idcs_cat == 2:     f_idcs = [0, 1]

        time_factor = opt_params["LoadupSamples_time_inc_factor"]
        ytime_train = np.tanh((ytree_train - 1.0) * time_factor) / 2.0 + 0.5
        ytime_test = np.tanh((ytree_test - 1.0) * time_factor) / 2.0 + 0.5

        scores = []

        mm = MachineModels(params=opt_params)
        for i in range(n_splits):
            tr_idx = tr_slice_list[i]
            te_idx = te_slice_list[i]

            # example: get date bounds if needed
            s_tr = slice(max(0, tr_idx.stop - n_training_days), tr_idx.stop)
            s_te = slice(te_idx.start, te_idx.stop)
            Xtr, ytr_time, ytr_tree = Xtime_train[s_tr], ytime_train[s_tr], ytree_train[s_tr]
            Xte, yte_time, yte_tree = Xtime_test[s_te], ytime_test[s_te], ytree_test[s_te]
            try:
                sc = _score_once_lstm(t_win, k, f_idcs, Xtr, ytr_tree, ytr_time, Xte, yte_tree, yte_time, mm,
                    device=device, do_transform=do_transform, quantile_val=quantile_val)
            except Exception as e:
                logger.info(f"Exception during scoring: {e}")
                sc = -np.inf
            scores.append(sc)

        vals = [v for v in scores if np.isfinite(v)]
        logger.info(f"Scores per splits: {vals}")
        vals = np.array(vals)
        vals_log = np.log(1.0 + vals)
        if len(vals) < (len(scores)//2):
            return 0.0
        return float(np.mean(vals_log)) if len(vals_log) else -np.inf
    return objective

In [9]:
studytime = 60*60*1
n_startup_trials = 5
studyname = f"optuna_clustering_idea_{formatted_str}"

In [10]:
# === Optuna driver ===
optuna.logging.enable_propagation()
sampler = optuna.samplers.TPESampler(n_startup_trials=n_startup_trials)
study = optuna.create_study(
    study_name=studyname,
    storage="sqlite:///sandbox_optuna.db",
    direction="maximize",
    load_if_exists=True,
    sampler=sampler,
)
study.optimize(make_objective(), timeout=studytime)

logger.info(f"Best parameters: {study.best_params}")
logger.info(f"Best score: {study.best_value}")

df: pd.DataFrame = study.trials_dataframe()
logger.info("\nTrials DataFrame:")
logger.info(df.sort_values("value").to_string())

param_importances = optuna.importance.get_param_importances(study)
logger.info("Parameter Importances:")
for key, value in param_importances.items():
    logger.info(f"{key}: {value}")

[I 2025-09-10 17:10:00,215] A new study created in RDB with name: optuna_clustering_idea_notebook-stomp-10sep25_1709


2025-09-10 17:10:00,215 - A new study created in RDB with name: optuna_clustering_idea_notebook-stomp-10sep25_1709


Epochs: 100%|██████████| 6/6 [00:00<00:00, 29.31it/s]

2025-09-10 17:10:02,343 - [LSTM] cluster 0: train=271, val_rmse=0.582130



Epochs: 100%|██████████| 6/6 [00:00<00:00, 60.56it/s]

...


Epochs:  33%|███▎      | 2/6 [00:00<00:00, 20.66it/s]
[W 2025-09-10 17:16:00,798] Trial 7 failed with parameters: {'CLUSTERS': 4, 'LoadupSamples_time_inc_factor': 51, 'LSTM_learning_rate': 0.0001929214937437397, 'LSTM_dropout': 0.00012562195148536464, 'LSTM_inter_dropout': 0.0068039875239076, 'LSTM_recurrent_dropout': 0.010364059376398142} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\KILightTouch\Desktop\RandomOdyssey\.venv\Lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\KILightTouch\AppData\Local\Temp\ipykernel_1376\633034923.py", line 48, in objective
    sc = _score_once_lstm(t_win, k, f_idcs, Xtr, ytr_tree, ytr_time, Xte, yte_tree, yte_time, mm,
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\KILightTouch\AppData\Local\Temp\ipykernel_1376\1427390912.p

2025-09-10 17:16:00,798 - Trial 7 failed with parameters: {'CLUSTERS': 4, 'LoadupSamples_time_inc_factor': 51, 'LSTM_learning_rate': 0.0001929214937437397, 'LSTM_dropout': 0.00012562195148536464, 'LSTM_inter_dropout': 0.0068039875239076, 'LSTM_recurrent_dropout': 0.010364059376398142} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\KILightTouch\Desktop\RandomOdyssey\.venv\Lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\KILightTouch\AppData\Local\Temp\ipykernel_1376\633034923.py", line 48, in objective
    sc = _score_once_lstm(t_win, k, f_idcs, Xtr, ytr_tree, ytr_time, Xte, yte_tree, yte_time, mm,
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\KILightTouch\AppData\Local\Temp\ipykernel_1376\1427390912.py", line 72, in _score_once_lstm
    model_c, info = mm.r

[W 2025-09-10 17:16:00,838] Trial 7 failed with value None.


2025-09-10 17:16:00,838 - Trial 7 failed with value None.


KeyboardInterrupt: 